# Dependencias

In [ ]:
# Instalamos una version vieja de torch para que sea compatible con la versión de los drivers de la GPU
import sys

!{sys.executable} -m pip uninstall -y torch torchvision torchaudio
!{sys.executable} -m pip cache purge
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
!{sys.executable} -m pip install torchinfo
!{sys.executable} -m pip install scikit-image

import torch
print("\n--- REPORTE FINAL ---")
print(f"Versión instalada de PyTorch: {torch.__version__}")

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
import time
import glob
import random
import pandas as pd
import copy
import re
import wrf
import cartopy.feature as cfeature
import cartopy.crs as crs

from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from tqdm.auto import tqdm
from datetime import datetime
from IPython.display import display
from matplotlib import animation
from skimage.metrics import structural_similarity as ssim
from wrf import getvar, get_cartopy, latlon_coords, to_np
from netCDF4 import Dataset

ruta_padre = os.path.abspath('..')
if ruta_padre not in sys.path:
    sys.path.append(ruta_padre)

CROP     = 1
archivos = sorted(glob.glob('../salidasWRF/wrfout_d01_*'))
with Dataset(archivos[0]) as ds0:
    ref = getvar(ds0, "LIGHTDENS")
    CART_PROJ = get_cartopy(ref)
    LATS, LONS = latlon_coords(ref)
LATS, LONS = to_np(LATS)[CROP:-CROP, CROP:-CROP], to_np(LONS)[CROP:-CROP, CROP:-CROP]

from modulos.tormentaDataset import TormentaDataset

# Chequeamos si hay GPU disponible para acelerar el entrenamiento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

In [ ]:
# Para REPRODUCIBILIDAD

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Baseline

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, in_channels=6):
        super(BaselineCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        
        self.salida = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        out = self.salida(x)
        return out

modelo_baseline = BaselineCNN(in_channels=6).to(device)
print(modelo_baseline)

In [ ]:
print("=== RESUMEN DE LA CNN BASELINE ===")
summary(modelo_baseline, input_size=(1, 6, 297, 297), 
        col_names=["input_size", "output_size", "num_params"])

# Unet

In [ ]:
class UNet_WRF(nn.Module):
    def __init__(self, in_channels=6):
        super(UNet_WRF, self).__init__()
        
        # --- ENCODER ---
        self.enc_conv1_1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.enc_conv1_2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.enc_conv2_1 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.enc_conv2_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        # --- BOTTLENECK ---
        self.bottleneck_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bottleneck_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        # --- DECODER ---
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec_conv1_1 = nn.Conv2d(128 + 64, 64, kernel_size=3, padding=1)
        self.dec_conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec_conv2_1 = nn.Conv2d(64 + 32, 32, kernel_size=3, padding=1)
        self.dec_conv2_2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)

        # --- SALIDA ---
        self.salida = nn.Conv2d(32, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()                     # SOLO para FACL

    def forward(self, x):
        x_pad = F.pad(x, (1, 2, 1, 2))  # 297 → 300

        # Encoder
        c1 = F.relu(self.enc_conv1_2(F.relu(self.enc_conv1_1(x_pad))))
        p1 = self.pool1(c1)
        
        c2 = F.relu(self.enc_conv2_2(F.relu(self.enc_conv2_1(p1))))
        p2 = self.pool2(c2)

        # Bottleneck
        bn = F.relu(self.bottleneck_2(F.relu(self.bottleneck_1(p2))))

        # Decoder
        u1 = self.up1(bn)
        d1 = F.relu(self.dec_conv1_2(F.relu(self.dec_conv1_1(torch.cat([u1, c2], dim=1)))))

        u2 = self.up2(d1)
        d2 = F.relu(self.dec_conv2_2(F.relu(self.dec_conv2_1(torch.cat([u2, c1], dim=1)))))

        out = self.salida(d2)
        out = self.sigmoid(out) # SOLO para FACL
        return out[:, :, 1:-2, 1:-2]  # 300 → 297

modelo_unet = UNet_WRF(in_channels=6).to(device)
print(modelo_unet)

In [ ]:
print("\n\n=== RESUMEN DE LA U-NET ===")
summary(modelo_unet, input_size=(1, 6, 297, 297), 
        col_names=["input_size", "output_size", "num_params"])

# Bob

In [ ]:
class BobModel(nn.Module):
    """
    Modelo de referencia que ignora completamente la entrada
    y siempre predice LIGHTDENS = 0 en todos los puntos de grilla.
    """
    def __init__(self):
        super(BobModel, self).__init__()
        self.register_buffer('valor_constante', torch.log(torch.tensor(0.1)))

    def forward(self, x):
        batch_size, _, H, W = x.shape
        salida = self.valor_constante.expand(batch_size, 1, H, W)
        return salida


modelo_bob = BobModel().to(device)
print(modelo_bob)

In [ ]:
print("\n\n=== RESUMEN DEL MODELO BOB ===")
summary(modelo_bob, input_size=(1, 6, 297, 297), 
        col_names=["input_size", "output_size", "num_params"])

In [ ]:
criterion_mse = nn.MSELoss()
modelo_bob.eval()

t_inicio = time.time()

train_loss_bob = 0.0
with torch.no_grad():
    for batch_X, batch_y in tqdm(train_loader, desc="Evaluando Modelo bob [Train]"):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        predicciones = modelo_bob(batch_X)
        loss = criterion_mse(predicciones, batch_y)
        train_loss_bob += loss.item() * batch_X.size(0)
train_loss_bob /= len(train_loader.dataset)

val_loss_bob = 0.0
with torch.no_grad():
    for batch_X, batch_y in tqdm(val_loader, desc="Evaluando Modelo bob [Val]"):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        predicciones = modelo_bob(batch_X)
        loss = criterion_mse(predicciones, batch_y)
        val_loss_bob += loss.item() * batch_X.size(0)
val_loss_bob /= len(val_loader.dataset)

tiempo_eval = time.time() - t_inicio

print(f"\nModelo bob — Train MSE: {train_loss_bob:.6f} | Val MSE: {val_loss_bob:.6f}")
print(f"Tiempo de evaluación: {tiempo_eval:.1f}s")

CARPETA_MODELOS = '../modelos'
os.makedirs(CARPETA_MODELOS, exist_ok=True)
fecha = datetime.now().strftime("%Y%m%d_%H%M")
ruta_guardado_bob = os.path.join(CARPETA_MODELOS, f"{fecha}_bob.pth")

torch.save({
    'model_state_dict': modelo_bob.state_dict(),
    'best_model_state_dict': modelo_bob.state_dict(),
    'historial_train_loss': [train_loss_bob],
    'historial_val_loss': [val_loss_bob],
    'historial_tiempo_epoch': [tiempo_eval],
    'mejor_epoca': 1,
    'mejor_val_loss': val_loss_bob,
    'epochs_entrenadas': 1,
    'lr': None,
    'patience': None,
}, ruta_guardado_bob)

# Entrenamiento

In [ ]:
# Ver el uso de las GPUs
!nvidia-smi

In [ ]:
# Para elegir la GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"¡Listo! Vamos a entrenar en: {device}")

In [ ]:
# ==========================================
# 1. EMPAQUETADO DESDE CARPETAS CON NORMALIZACIÓN
# ==========================================
ruta_train = '../dataset/train'
ruta_val = '../dataset/val'
ruta_stats = '../dataset/stats.npz'

print("Mapeando archivos y cargando estadísticas globales (Min-Max)...")

# Instanciamos los datasets pasándoles la carpeta de datos y las estadísticas
train_dataset = TormentaDataset(split_dir=ruta_train, stats_path=ruta_stats)
val_dataset = TormentaDataset(split_dir=ruta_val, stats_path=ruta_stats)

BATCH_SIZE = 32

# Creamos los loaders
generator = torch.Generator()
generator.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f"¡DataLoaders listos y normalizados!")
print(f"-> Archivos para entrenar (Train): {len(train_dataset)}")
print(f"-> Archivos para validar (Val): {len(val_dataset)}")
print(f"-> Batches por época (Train): {len(train_loader)}")

In [ ]:
# ==========================================
# 2. FUNCIONES DE PÉRDIDA
# ==========================================
def loss_ponderado(y_real_log, y_pred_log, peso_positivo=20.0):
    """
    MSE ponderado: penaliza con mayor peso los píxeles donde hay actividad
    eléctrica real (y_real > 0), para compensar el fuerte desbalance espacial.
    """
    # Desnormalizar a escala real: y = log(x+0.1) → x = exp(y) - 0.1
    y_real = torch.exp(y_real_log) - 0.1

    diferencia = (y_real_log - y_pred_log) ** 2
    pesos = torch.where(y_real > 0,
                         torch.tensor(peso_positivo, device=y_real_log.device),
                         torch.tensor(1.0, device=y_real_log.device))
    return torch.mean(diferencia * pesos)

# Basado en https://github.com/argenycw/FACL
def fourier_amplitude_loss(X, X_hat):
    F     = torch.fft.fft2(X,     norm='ortho')
    F_hat = torch.fft.fft2(X_hat, norm='ortho')
    return torch.mean((torch.abs(F) - torch.abs(F_hat)) ** 2)

# Basado en https://github.com/argenycw/FACL
def fourier_correlation_loss(X, X_hat, eps=1e-8):
    F     = torch.fft.fft2(X,     norm='ortho')
    F_hat = torch.fft.fft2(X_hat, norm='ortho')
    dims  = tuple(range(1, F.dim()))  # todo menos el batch

    numerador  = torch.sum((F * torch.conj(F_hat)).real, dim=dims)
    norm_F     = torch.sqrt(torch.sum(torch.abs(F) ** 2, dim=dims))
    norm_Fhat  = torch.sqrt(torch.sum(torch.abs(F_hat) ** 2, dim=dims))

    correlacion = numerador / (norm_F * norm_Fhat + eps)
    return (1 - correlacion).mean()

# Extraído de train_utils_fourier.py
class RandomFourierLoss(nn.Module):
    """
    Fourier Amplitude and Correlation Loss (FACL).

    Implementa la función de pérdida propuesta por Yan et al. (2024),
    alternando aleatoriamente entre:

        - FAL (Fourier Amplitude Loss)
        - FCL (Fourier Correlation Loss)

    La probabilidad de utilizar FAL aumenta gradualmente durante el
    entrenamiento, reproduciendo la estrategia del artículo original:

        • al comienzo domina FCL, que enseña la estructura global;
        • luego FAL aparece cada vez más seguido para afinar amplitudes
          de alta frecuencia y mejorar la nitidez.

    Referencia:
        https://arxiv.org/abs/2410.23159
    """
    def __init__(self, min_prob=0.0, max_prob=0.95, prob_slope=4.0e-4):
        super(RandomFourierLoss, self).__init__()
        self.step = 0
        self.min_prob = min_prob
        self.max_prob = max_prob
        self.prob_slope = prob_slope

    def fcl(self, fft_pred, fft_truth, eps=1e-8):
        """
        Fourier Correlation Loss (FCL).

        Mide la correlación normalizada entre las transformadas de Fourier
        de la predicción y del ground truth.

        Matemáticamente, para cada imagen i del batch:

            FCL_i = 1 - Re(<F_pred_i, conj(F_truth_i)>) /
                        (||F_pred_i|| · ||F_truth_i||)

        donde Re(·) extrae la parte real del producto interno complejo.
        El valor final es el promedio de FCL_i sobre todas las imágenes del batch.

        Esta pérdida depende de la estructura global de la imagen
        (fase + organización espacial), pero es prácticamente
        invariante ante cambios globales de intensidad.

        Parámetros
        ----------
        fft_pred : Tensor complejo
            FFT de la predicción.

        fft_truth : Tensor complejo
            FFT del ground truth.

        eps : float, default 1e-8
            Constante de estabilidad numérica para evitar división por cero.

        Returns
        -------
        Tensor escalar
            Valor promedio de la Fourier Correlation Loss sobre el batch.
        """
        dims = tuple(range(1, fft (salvo los bordes). Los bordes _pred.dim()))
        numerador   = torch.sum((fft_pred * torch.conj(fft_truth)).real, dim=dims)
        norm_pred   = torch.sqrt(torch.sum(fft_pred.abs() ** 2, dim=dims))
        norm_truth  = torch.sqrt(torch.sum(fft_truth.abs() ** 2, dim=dims))
        correlacion = numerador / (norm_pred * norm_truth + eps)
        return (1 - correlacion).mean()

    def fal(self, fft_pred, fft_truth):
        """
        Fourier Amplitude Loss (FAL).

        Compara únicamente el espectro de amplitudes de ambas imágenes,
        ignorando completamente la fase.

        Matemáticamente:

            FAL = mean((|F_pred| - |F_gt|)^2)

        Esta pérdida favorece que la distribución de frecuencias sea la
        correcta, produciendo imágenes más nítidas. Sin embargo, por sí
        sola no puede reconstruir correctamente la estructura espacial,
        ya que la fase del ground truth no interviene en el gradiente.

        Parámetros
        ----------
        fft_pred : Tensor complejo
            FFT de la predicción.

        fft_truth : Tensor complejo
            FFT del ground truth.

        Returns
        -------
        Tensor escalar
            Valor de la Fourier Amplitude Loss.
        """
        return ((fft_pred.abs() - fft_truth.abs()) ** 2).mean()

    def forward(self, pred, gt, mode='train'):
        """
        Calcula la pérdida FACL.

        Durante el entrenamiento se selecciona aleatoriamente entre FCL
        y FAL. La probabilidad de elegir FAL aumenta linealmente con el
        número de iteraciones.

        Durante validación se utiliza una mezcla fija 50%-50%.

        Finalmente la pérdida se multiplica por sqrt(H·W), tal como
        propone el código oficial, para mantener una escala comparable
        entre imágenes de distinto tamaño.

        Parámetros
        ----------
        pred : Tensor
            Predicción del modelo con forma (N,1,H,W).

        gt : Tensor
            Ground truth con forma (N,1,H,W).

        mode : {"train", "val"}
            Modo de funcionamiento.

        Returns
        -------
        Tensor escalar
            Valor final de la pérdida.
        """
        pred = pred.squeeze(1)  # (N,1,H,W) → (N,H,W); fijo el dim=1 para no arriesgar squeezear el batch si N=1
        gt = gt.squeeze(1)
        fft_pred = torch.fft.fftn(pred, dim=[-1, -2], norm='ortho')
        fft_gt = torch.fft.fftn(gt, dim=[-1, -2], norm='ortho')

        prob_t = (self.step * self.prob_slope) * (self.max_prob - self.min_prob) + self.min_prob
        prob_t = min(prob_t, self.max_prob)
        prob = 1.0 if np.random.rand() < prob_t else 0.0
        if mode == 'val':
            prob = 0.5  # Para validación, mitad y mitad

        H, W = pred.shape[-2:]
        weight = np.sqrt(H * W)
        loss = prob * self.fal(fft_pred, fft_gt) + (1 - prob) * self.fcl(fft_pred, fft_gt)
        loss = loss * weight

        if mode == 'train':
            self.step += 1
        return loss

In [ ]:
# =====================================================================
# 3. FUNCIÓN DE CONFIGURACIÓN GLOBAL DE ENTRENAMIENTO
# =====================================================================

def configurar_entrenamiento(tipo_modelo='unet', loss='bmse', lr=0.001, epocas=30, patience=5,
                              min_prob=0.0, max_prob=0.95, prob_slope=4.0e-4):
    """
    Configura y resetea el entorno de entrenamiento definiendo e inicializando las variables
    globales necesarias para el loop de optimización de PyTorch.

    Parameteros
    ----------
    tipo_modelo : str, default 'unet'
        Determina la arquitectura del modelo que se enviará al dispositivo (GPU/CPU).
        Opciones válidas:
            * 'unet' : Configura la arquitectura U-Net Completa.
            * 'baseline' : Configura la arquitectura CNN Simple.
    loss : str, default 'bmse'
        Función de pérdida utilizada durante el entrenamiento.

        Opciones válidas:
            * 'bmse' : Utiliza la función `loss_ponderado`
              (Weighted Binary Cross Entropy con peso positivo 20).

            * 'facl' : Utiliza la función `RandomFourierLoss`
              propuesta por Yan et al. (2024), alternando entre
              Fourier Correlation Loss (FCL) y Fourier Amplitude
              Loss (FAL).
    lr : float, default 0.001
        Tasa de aprendizaje (Learning Rate) inicial que utilizará el optimizador Adam 
        para ajustar los pesos de la red neuronal durante el backward pass.
    epocas : int, default 30
        Número total de iteraciones completas sobre el dataset de entrenamiento (Épocas) 
        que ejecutará el loop principal.
    patience : int, default 5
        Cantidad de épocas sin mejora en validación antes de detener el entrenamiento (Early Stopping).
    min_prob : float, default 0.0
        Probabilidad inicial de seleccionar Fourier Amplitude Loss (FAL) al utilizar
        `RandomFourierLoss`. Durante las primeras iteraciones domina FCL.
    max_prob : float, default 0.95
        Probabilidad máxima que puede alcanzar la selección de FAL durante el
        entrenamiento. Una vez alcanzado este valor, la probabilidad permanece fija.
    prob_slope : float, default 4e-4
        Pendiente del incremento lineal de la probabilidad de utilizar FAL en
        `RandomFourierLoss`. Valores mayores hacen que el entrenamiento pase antes
        de priorizar el refinamiento de altas frecuencias.

    Returns
    -------
    None
        La función no retorna ningún objeto. En su lugar, inicializa o pisa las 
        siguientes variables en el entorno global del notebook:
            * `modelo_actual`: El objeto del modelo elegido, montado en el `device`.
            * `LEARNING_RATE`: El valor numérico del learning rate asignado.
            * `EPOCHS`: La cantidad total de épocas configuradas.
            * `PATIENCE`: Cantidad de épocas de tolerancia sin mejoras para el Early Stopping.
            * `criterion`: La función de costo/pérdida a optimizar (por defecto, loss_ponderado).
            * `optimizer`: El optimizador Adam vinculado a los parámetros de `modelo_actual`.
            * `historial_train_loss`: Una lista vacía `[]` para almacenar pérdidas de Train.
            * `historial_val_loss`: Una lista vacía `[]` para almacenar pérdidas de Validación.
            * `historial_tiempo_epoch`: Una lista vacía `[]` para registrar cuánto tarda cada época.
            * `NOMBRE_MODELO`: Un string con el nombre normalizado del modelo (ej. 'unet' o 'baseline').
    """
    global modelo_actual, LEARNING_RATE, EPOCHS, PATIENCE, criterion, optimizer, scheduler
    global historial_train_loss, historial_val_loss, historial_tiempo_epoch, NOMBRE_MODELO
    NOMBRE_MODELO = tipo_modelo.lower().strip()
    
    # 1. SELECCIÓN DEL MODELO
    if NOMBRE_MODELO == 'unet':
        modelo_actual = UNet_WRF().to(device)
        nombre_impresion = "U-Net Completa"
            
    elif NOMBRE_MODELO == 'baseline':
        modelo_actual = BaselineCNN().to(device)
        nombre_impresion = "CNN Simple (Baseline)"
            
    else:
        raise ValueError(f"Error: '{tipo_modelo}' no es una opción válida de arquitectura.")

    # 2. ASIGNACIÓN DE HIPERPARÁMETROS GLOBALES
    LEARNING_RATE = lr
    EPOCHS = epocas
    PATIENCE = patience

    # 3. CONFIGURACIÓN DE COMPONENTES DE OPTIMIZACIÓN (optimizador, loss y scheduler)
    optimizer = optim.Adam(modelo_actual.parameters(), lr=LEARNING_RATE)
    
    nombre_loss = loss.lower().strip()

    if nombre_loss == "bmse":
        criterion = loss_ponderado
        descripcion_loss = "loss_ponderado (Peso Positivo=20)"

    elif nombre_loss == "facl":
        criterion = RandomFourierLoss(
            min_prob=min_prob,
            max_prob=max_prob,
            prob_slope=prob_slope
        ).to(device)

        descripcion_loss = (
            f"RandomFourierLoss "
            f"(min_prob={min_prob}, "
            f"max_prob={max_prob}, "
            f"prob_slope={prob_slope})"
        )

    else:
        raise ValueError(
            f"Error: '{loss}' no es una función de pérdida válida."
        )

    # T_max = épocas totales → el LR decae suavemente desde lr hasta ~0
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-7
    )
    
    # 4. RESETEO COMPLETO DE HISTORIALES MÉTRICOS
    historial_train_loss = []
    historial_val_loss = []
    historial_tiempo_epoch = []

    # --- REPORTE DE CONFIRMACIÓN POR PANTALLA ---
    print("==================================================")
    print("  CONFIGURACIÓN GLOBAL DE ENTRENAMIENTO EXITOSA   ")
    print("==================================================")
    print(f"-> Modelo Activo:  {nombre_impresion}")
    print(f"-> Corriendo en:   {device}")
    print(f"-> Épocas totales: {EPOCHS}")
    print(f"-> Patience (ES):  {PATIENCE} épocas")
    print(f"-> Learning Rate:  {LEARNING_RATE}")
    print(f"-> Función Loss:   {descripcion_loss}")
    print("==================================================")

In [ ]:
configurar_entrenamiento(tipo_modelo='baseline', lr=0.0005, epocas=200, patience=20)

In [ ]:
configurar_entrenamiento(tipo_modelo='unet', loss='facl', lr=1e-4, epocas=200, patience=40, prob_slope=1e-5, max_prob=0.7)

In [ ]:
# ==========================================
# 4. BUCLE DE ENTRENAMIENTO
# ==========================================
CARPETA_MODELOS = '../modelos'
mejor_val_loss = float('inf')
epocas_sin_mejora = 0
mejor_pesos = None
mejor_epoca = 0

os.makedirs(CARPETA_MODELOS, exist_ok=True)
fecha = datetime.now().strftime("%Y%m%d_%H%M")
ruta_guardado = os.path.join(CARPETA_MODELOS, f"{fecha}_{NOMBRE_MODELO}.pth")

print(f"--- INICIANDO ENTRENAMIENTO (máx. {EPOCHS} épocas, patience={PATIENCE}) ---")
tiempo_inicio_total = time.time()
torch.cuda.empty_cache()

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # ==========================================
    # FASE DE ENTRENAMIENTO
    # ==========================================
    modelo_actual.train() 
    train_loss_acumulada = 0.0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [Train]", leave=False)
    
    for batch_X, batch_y in train_pbar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()                         
        predicciones = modelo_actual(batch_X)         
        loss = criterion(predicciones, batch_y, mode='train')       
        loss.backward()                               
        optimizer.step()                              
        
        train_loss_acumulada += loss.item() * batch_X.size(0)
        
        # Actualizamos el texto a la derecha de la barra para ver el loss en tiempo real
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    epoch_train_loss = train_loss_acumulada / len(train_loader.dataset)
    historial_train_loss.append(epoch_train_loss)
    
    # ==========================================
    # FASE DE VALIDACIÓN
    # ==========================================
    modelo_actual.eval()
    val_loss_acumulada = 0.0
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [Val]  ", leave=False)
    
    with torch.no_grad(): 
        for batch_X, batch_y in val_pbar:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            predicciones = modelo_actual(batch_X)
            loss = criterion(predicciones, batch_y, mode='val')
            
            val_loss_acumulada += loss.item() * batch_X.size(0)
            val_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    epoch_val_loss = val_loss_acumulada / len(val_loader.dataset)
    historial_val_loss.append(epoch_val_loss)
    tiempo_epoch = time.time() - start_time
    historial_tiempo_epoch.append(tiempo_epoch)

    # ==========================================
    # EARLY STOPPING
    # ==========================================
    if epoch_val_loss < mejor_val_loss:
        mejor_val_loss = epoch_val_loss
        epocas_sin_mejora = 0
        mejor_epoca = epoch + 1
        # Guardamos una copia en memoria de los mejores pesos
        mejor_pesos = copy.deepcopy(modelo_actual.state_dict())
    else:
        epocas_sin_mejora += 1
    
    # ==========================================
    # GUARDADO
    # ==========================================
    torch.save({
        'model_state_dict': modelo_actual.state_dict(),
        'best_model_state_dict': mejor_pesos,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict':  scheduler.state_dict(),
        'historial_train_loss': historial_train_loss,
        'historial_val_loss': historial_val_loss,
        'historial_tiempo_epoch': historial_tiempo_epoch,
        'mejor_epoca': mejor_epoca,
        'mejor_val_loss': mejor_val_loss,
        'epochs_entrenadas': epoch + 1,
        'lr': LEARNING_RATE,
        'patience': PATIENCE,
    }, ruta_guardado)

    # ==========================================
    # SCHEDULER STEP
    # ==========================================
    scheduler.step()
    lr_actual = optimizer.param_groups[0]['lr']
    
    # ==========================================
    # REPORTE DE LA ÉPOCA
    # ==========================================
    tiempo_epoch = time.time() - start_time
    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Tiempo: {tiempo_epoch:.2f}s | "
          f"Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f} | "
          f"Sin mejora: {epocas_sin_mejora}/{PATIENCE}")
    
    # ==========================================
    # CORTE POR EARLY STOPPING
    # ==========================================
    if epocas_sin_mejora >= PATIENCE:
        print(f"\nEarly stopping soloactivado en la época {epoch+1}.")
        print(f"Mejor val_loss: {mejor_val_loss:.6f} (época {mejor_epoca})")
        break

# ==========================================
# RESTAURAR MEJORES PESOS
# ==========================================
modelo_actual.load_state_dict(mejor_pesos)
print(f"\nPesos restaurados a la mejor época ({mejor_epoca}).")

# ==========================================
# REPORTE FINAL
# ==========================================
tiempo_fin_total = time.time()
segundos_totales = tiempo_fin_total - tiempo_inicio_total
horas_totales = segundos_totales / 3600
horas_enteras = int(horas_totales)
minutos_enteros = int((segundos_totales % 3600) / 60)

print("\n¡Entrenamiento finalizado exitosamente!")
print(f"Modelo final disponible en: {ruta_guardado}")
print(f"Tiempo total: {horas_totales:.2f} horas ({horas_enteras}h {minutos_enteros}m)")
print(f"Tiempo promedio por época: {np.mean(historial_tiempo_epoch):.1f}s")
print(f"Mejor época: {mejor_epoca} | Mejor val_loss: {mejor_val_loss:.6f}")

# Resultados

In [ ]:
def resumen_entrenamiento(checkpoint, nombre_modelo, train_loader, val_loader, batch_size):
    """
    Imprime un resumen del entrenamiento combinando datos del checkpoint
    (guardados durante el entrenamiento) con datos del DataLoader actual
    (que no se guardan en el .pth).
    """
    tiempo_total_seg = sum(checkpoint['historial_tiempo_epoch'])
    horas = int(tiempo_total_seg // 3600)
    minutos = int((tiempo_total_seg % 3600) // 60)
    tiempo_str = f"{horas}h {minutos}m" if horas > 0 else f"{minutos}m"

    print(f"-> Modelo Activo:  {nombre_modelo}")
    print(f"-> Épocas totales: {checkpoint['epochs_entrenadas']}")
    print(f"-> Learning Rate:  {checkpoint['lr']}")
    print(f"-> Batch Size:     {batch_size}")
    print(f"-> Archivos para entrenar (Train): {len(train_loader.dataset)}")
    print(f"-> Archivos para validar (Val): {len(val_loader.dataset)}")
    print(f"-> Batches por época (Train): {len(train_loader)}")
    print(f"-> Tiempo: {tiempo_str}")
    print(f"-> Mejor val_loss: {checkpoint['mejor_val_loss']:.6f} (época {checkpoint['mejor_epoca']})")
    print(f"-> Patience: {checkpoint['patience']}")

def generar_ruta_imagen(carpeta, prefijo, idx=None):
    """
    Genera la ruta de guardado. Si se provee un 'idx', lo incluye en el nombre.
    Si el archivo ya existe, le anexa un contador (_01, _02, etc.) para no pisarlo.
    """
    os.makedirs(carpeta, exist_ok=True)
    
    # Armamos el nombre base. Ej: "reconstruccion_unet_idx_42" o "curva_error_unet"
    if idx is not None:
        nombre_base = f"{prefijo}_idx_{idx}"
    else:
        nombre_base = prefijo
        
    ruta = os.path.join(carpeta, f"{nombre_base}.png")
    
    # Si no existe, usamos ese nombre directamente
    if not os.path.exists(ruta):
        return ruta
        
    # Si ya existe, empezamos a probar con un contador hasta encontrar uno libre
    contador = 1
    while True:
        ruta_con_contador = os.path.join(carpeta, f"{nombre_base}_{contador:02d}.png")
        if not os.path.exists(ruta_con_contador):
            return ruta_con_contador
        contador += 1


def graficar_curva_error(checkpoint, nombre_modelo='Modelo'):
    """
    Grafica la curva de MSE para train y validación a partir de un checkpoint cargado.
    """
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    mejor_epoca    = checkpoint['mejor_epoca']
    mejor_val_loss = checkpoint['mejor_val_loss']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epocas, historial_train, 'o-', label='Train Loss (FACL)', color='steelblue')
    ax.plot(epocas, historial_val,   'o-', label='Val Loss (FACL)',   color='#722F37')
    
    ax.scatter([mejor_epoca], [mejor_val_loss], s=180, marker='*',
               color='gold', edgecolor='black', zorder=5,
               label=f'Mejor época ({mejor_epoca})')
    ax.axvline(mejor_epoca, color='gray', linestyle='--', alpha=0.5)
    ax.annotate(f'{mejor_val_loss:.4f}',
                xy=(mejor_epoca, mejor_val_loss),
                xytext=(8, 10), textcoords='offset points',
                fontsize=9, fontweight='bold')
    
    ax.set_xlabel('Época')
    ax.set_ylabel('Weighted MSE')
    ax.set_title(f'Curva de aprendizaje — {nombre_modelo}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"curva_error_{nombre_limpio}")
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica de error guardada en: {ruta_guardado}")
    
    plt.show()
    
    mejor_epoca = historial_val.index(min(historial_val)) + 1
    print(f"Mejor val_loss: {min(historial_val):.6f} (época {mejor_epoca})")


def graficar_reconstruccion(modelo, dataset, nombre_modelo='Modelo', canal_entrada=0, idx=None, vmax_percentil=99):
    """
    Muestra la variable de entrada, el target real y la predicción del modelo
    para un timestep aleatorio (o el índice especificado) del dataset.

    Parámetros
    ----------
    modelo        : modelo PyTorch ya cargado y en eval mode
    dataset       : TormentaDataset (train o val)
    nombre_modelo : str, para el título
    canal_entrada : int, índice del canal a mostrar como entrada
    idx           : int o None. Si None, elige uno al azar.
    """
    VAR_NOMBRES = ['MDBZ', 'HIELO', 'AGUA', 'CTT', 'LPI', 'WMAX']

    if idx is None:
        idx = random.randint(0, len(dataset) - 1)
    print(f"Timestep seleccionado: {idx} ({dataset.files[idx].name})")

    modelo.eval()
    X, y = dataset[idx]
    with torch.no_grad():
        pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

    # Desnormalizar target y predicción
    y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
    y_pred = dataset.desnormalizar_y(pred_log).numpy()
    entrada = X[canal_entrada].numpy()

    # Escala basada en percentil (ignora picos extremos)
    vmax_real = np.percentile(y_real[y_real > 0], vmax_percentil) if (y_real > 0).any() else 1

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'{nombre_modelo} — Timestep {idx}', fontsize=13)

    ax0 = fig.add_subplot(1, 3, 1, projection=CART_PROJ)
    ax0.add_feature(cfeature.BORDERS, linewidth=0.8)
    ax0.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax0.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
    cf0 = ax0.contourf(LONS, LATS, entrada, transform=crs.PlateCarree(), cmap='Blues')
    ax0.set_title(f'Entrada: {VAR_NOMBRES[canal_entrada]}')
    fig.colorbar(cf0, ax=ax0, fraction=0.046, pad=0.04)

    ax1 = fig.add_subplot(1, 3, 2, projection=CART_PROJ)
    ax1.add_feature(cfeature.BORDERS, linewidth=0.8)
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax1.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
    cf1 = ax1.contourf(LONS, LATS, y_real, transform=crs.PlateCarree(),
                        cmap='hot_r', levels=np.linspace(0, vmax_real, 21), vmin=0, vmax=vmax_real, extend='max')
    ax1.set_title('Real: LIGHTDENS')
    fig.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04)

    ax2 = fig.add_subplot(1, 3, 3, projection=CART_PROJ)
    ax2.add_feature(cfeature.BORDERS, linewidth=0.8)
    ax2.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax2.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
    cf2 = ax2.contourf(LONS, LATS, y_pred, transform=crs.PlateCarree(),
                        cmap='hot_r', levels=np.linspace(0, vmax_real, 21), vmin=0, vmax=vmax_real, extend='max')
    ax2.set_title(f'Predicción: {nombre_modelo}')
    fig.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04)

    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"reconstruccion_{nombre_limpio}", idx=idx)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Reconstrucción guardada en: {ruta_guardado}")
    
    plt.show()

    print(f"Real  — min: {y_real.min():.2f}, max: {y_real.max():.2f}, mean: {y_real.mean():.4f}")
    print(f"Pred  — min: {y_pred.min():.2f}, max: {y_pred.max():.2f}, mean: {y_pred.mean():.4f}")


def animar_reconstruccion(modelo, dataset, nombre_modelo='Modelo', fps=5, vmax_percentil=99):
    """
    Genera una animación comparando la predicción del modelo contra el valor real
    de LIGHTDENS para todos los timesteps del dataset (pensado para el conjunto
    de validación, que contiene un día completo).

    Parámetros
    ----------
    modelo        : modelo PyTorch ya cargado y en eval mode
    dataset       : TormentaDataset (normalmente val_dataset)
    nombre_modelo : str, para el título y el nombre de archivo
    fps           : cuadros por segundo de la animación
    """
    modelo.eval()
    n_frames = len(dataset)

    # --- 1. Calcular vmax global (vmin siempre es 0 porque LIGHTDENS no es negativo) ---
    vmax = 0.0
    predicciones_cache = []  # Cacheamos para no correr el modelo dos veces
    reales_cache = []

    for idx in tqdm(range(n_frames), desc=f"Calculando rango — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        y_pred = dataset.desnormalizar_y(pred_log).numpy()

        reales_cache.append(y_real)
        predicciones_cache.append(y_pred)

    valores_activos = np.concatenate([r[r > 0] for r in reales_cache if (r > 0).any()])
    vmax = np.percentile(valores_activos, vmax_percentil) if len(valores_activos) > 0 else 1

    print(f"Rango LIGHTDENS: 0.00 - {vmax:.2f}")

    # --- 2. Extraer fecha del primer archivo del dataset ---
    nombre_archivo = dataset.files[0].name
    match = re.search(r'(\d{4}-\d{2}-\d{2})', nombre_archivo)
    fecha_val = match.group(1) if match else "fecha_desconocida"

    # --- 3. Preparar figura ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': CART_PROJ})
    fig.suptitle(f'{nombre_modelo} — Validación {fecha_val}', fontsize=14)

    pbar = tqdm(total=n_frames, desc="Generando frames")

    def update(frame_idx):
        for ax in axes:
            ax.clear()
            ax.add_feature(cfeature.BORDERS, linewidth=0.8)
            ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
            ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')

        cf0 = axes[0].contourf(LONS, LATS, reales_cache[frame_idx], transform=crs.PlateCarree(),
                               cmap='hot_r', levels=np.linspace(0, vmax, 21), vmin=0, vmax=vmax, extend='max')
        axes[0].set_title(f'Real: LIGHTDENS — frame {frame_idx}')

        cf1 = axes[1].contourf(LONS, LATS, predicciones_cache[frame_idx], transform=crs.PlateCarree(),
                               cmap='hot_r', levels=np.linspace(0, vmax, 21), vmin=0, vmax=vmax, extend='max')
        axes[1].set_title(f'Predicción: {nombre_modelo}')

        pbar.update(1)
        return cf0, cf1

    # --- 4. Colorbar único y animación ---
    cf_inicial_real, cf_inicial_pred = update(0)
    fig.colorbar(cf_inicial_real, ax=axes[0], label='LIGHTDENS (flash/columna)', fraction=0.046, pad=0.04)
    fig.colorbar(cf_inicial_pred, ax=axes[1], label='LIGHTDENS (flash/columna)', fraction=0.046, pad=0.04)

    ani = animation.FuncAnimation(fig, update,
                                   frames=range(1, n_frames),
                                   interval=200)

    # --- 5. Guardado ---
    carpeta_destino = "../videos&gifs"
    os.makedirs(carpeta_destino, exist_ok=True)
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    nombre_archivo_salida = f"reconstruccion_{nombre_limpio}_{fecha_val}.gif"
    ruta_final = os.path.join(carpeta_destino, nombre_archivo_salida)

    ani.save(ruta_final, writer="pillow", fps=fps)
    pbar.close()
    print(f"Animación guardada en {ruta_final}")
    plt.show()

def mostrar_historial_tabular(checkpoint):
    """
    Toma el checkpoint de un modelo y muestra una tabla interactiva con 
    el historial de pérdidas (Train y Val) época por época.
    Resalta en verde la época con el menor error de validación.
    """
    # 1. Extraemos las listas del checkpoint
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    
    # 2. Armamos un DataFrame de Pandas
    df = pd.DataFrame({
        'Época': epocas,
        'Train Loss (WMSE)': historial_train,
        'Val Loss (WMSE)': historial_val
    })
    
    # 3. Calculamos la diferencia con la época anterior para ver si mejoró
    mejor_hasta_ayer = df['Val Loss (WMSE)'].shift(1).cummin().fillna(float('inf'))
    df['¿Mejoró?'] = (df['Val Loss (WMSE)'] < mejor_hasta_ayer).apply(lambda x: '🟢 Sí' if x else '🔴 No')
    df.loc[0, '¿Mejoró?'] = '🟢 Sí'
    
    # 4. Buscamos cuál fue la mejor época absoluta
    mejor_val_loss = min(historial_val)
    
    # 5. Función para pintar la fila ganadora de verde
    def resaltar_mejor_epoca(fila):
        if fila['Val Loss (WMSE)'] == mejor_val_loss:
            return ['background-color: #d4edda; color: #155724; font-weight: bold'] * len(fila)
        else:
            return [''] * len(fila)
            
    # 6. Aplicamos el formato (dejamos 6 decimales) y mostramos la tabla
    tabla_estilizada = df.style.apply(resaltar_mejor_epoca, axis=1) \
                               .format({'Train Loss (WMSE)': "{:.6f}", 
                                        'Val Loss (WMSE)': "{:.6f}"}) \
                               .hide(axis="index")
    
    print("HISTORIAL DE ENTRENAMIENTO TABULAR")
    print("-" * 40)
    display(tabla_estilizada)

    return df

def calcular_data_range_global(dataset):
    """
    Calcula el rango (max - min) de LIGHTDENS real sobre TODO el dataset.
    """
    max_global = float("-inf")
    min_global = float("inf")
    for idx in tqdm(range(len(dataset)), desc="Calculando data_range global"):
        _, y = dataset[idx]
        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        max_global = max(max_global, y_real.max())
        min_global = min(min_global, y_real.min())
    return max_global - min_global

def calcular_metricas_timestep(y_real, y_pred, umbral=0.0, data_range=None):
    """
    Calcula un set de métricas de verificación comparando la predicción
    contra LIGHTDENS real, para un único timestep.

    Parámetros
    ----------
    y_real, y_pred : np.ndarray 2D
    umbral         : float, valor de LIGHTDENS a partir del cual se considera
                     "evento" para las métricas binarias (HP, CSI).

    Returns
    -------
    dict con: mse, rmse, ssim
    """
    # --- Métricas continuas ---
    error = y_real - y_pred
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))

    if data_range is None:
        data_range = max(y_real.max(), y_pred.max()) - min(y_real.min(), y_pred.min())
    valor_ssim = ssim(y_real, y_pred, data_range=data_range) if data_range > 0 else 1.0

    # --- Métricas binarias (tabla de contingencia) ---
    # real_bin = y_real > umbral
    # pred_bin = y_pred > umbral

    # hits           = np.sum(real_bin & pred_bin)
    # misses         = np.sum(real_bin & ~pred_bin)
    # falsas_alarmas = np.sum(~real_bin & pred_bin)

    # hp  = hits / (hits + misses) if (hits + misses) > 0 else np.nan
    # csi = hits / (hits + misses + falsas_alarmas) if (hits + misses + falsas_alarmas) > 0 else np.nan

    # return {'mse': mse, 'rmse': rmse, 'mae': mae, 'ssim': valor_ssim, 'hp': hp, 'csi': csi}
    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'ssim': valor_ssim}

def evaluar_dataset(modelo, dataset, umbral=0.0, data_range=None):
    """
    Recorre todo el dataset, calcula las métricas de verificación para cada
    timestep y las devuelve en un DataFrame (una fila por timestep).
    """
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    for idx in tqdm(range(len(dataset)), desc="Calculando métricas"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        y_pred = dataset.desnormalizar_y(pred_log).numpy()

        metricas = calcular_metricas_timestep(y_real, y_pred, umbral=umbral, data_range=data_range)
        metricas['idx'] = idx
        registros.append(metricas)

    return pd.DataFrame(registros)

def graficar_metrica(df_metricas, columna, nombre_modelo='Modelo', ylabel=None, color='#722F37'):
    """
    Grafica la evolución de una métrica de verificación a lo largo de los
    timesteps del conjunto de validación, con la media como referencia.
    """
    serie = df_metricas[['idx', columna]].dropna(subset=[columna]) # <-- ignora NaN
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(serie['idx'], serie[columna], 'o-', color=color, markersize=4)

    media = serie[columna].mean()
    ax.axhline(media, color='gray', linestyle='--', alpha=0.6, label=f'Media: {media:.3f}')

    ax.set_xlabel('Timestep (val)')
    ax.set_ylabel(ylabel or columna.upper())
    ax.set_title(f'{ylabel or columna.upper()} — {nombre_modelo}')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"{columna}_{nombre_limpio}")
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica de {columna.upper()} guardada en: {ruta_guardado}")

    plt.show()
    return media

def graficar_mse(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'mse', nombre_modelo, ylabel='MSE')

def graficar_rmse(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'rmse', nombre_modelo, ylabel='RMSE (flashes/columna)')

def graficar_mae(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'mae', nombre_modelo, ylabel='MAE')

def graficar_ssim(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'ssim', nombre_modelo, ylabel='SSIM')

# def graficar_hp(df, nombre_modelo='Modelo'):
#     return graficar_metrica(df, 'hp', nombre_modelo, ylabel='HP (Probabilidad de detección)')

# def graficar_csi(df, nombre_modelo='Modelo'):
#     return graficar_metrica(df, 'csi', nombre_modelo, ylabel='CSI (Critical Success Index)')

def calcular_r2_global(modelo, dataset):
    """
    Calcula R² como métrica global sobre todos los píxeles activos
    del dataset.
    """
    modelo.eval()
    y_real_total = []
    y_pred_total = []

    for idx in tqdm(range(len(dataset)), desc="Calculando R² global"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        y_pred = dataset.desnormalizar_y(pred_log).numpy()

        mascara = y_real > 0
        if mascara.sum() > 0:
            y_real_total.append(y_real[mascara])
            y_pred_total.append(y_pred[mascara])

    y_real_all = np.concatenate(y_real_total)
    y_pred_all = np.concatenate(y_pred_total)

    ss_res = np.sum((y_real_all - y_pred_all) ** 2)
    ss_tot = np.sum((y_real_all - y_real_all.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    print(f"R² global (sobre {len(y_real_all):,} píxeles activos): {r2:.4f}")
    return r2

def graficar_r2_comparativo(r2_dict, nombre_figura='r2_comparativo'):
    """
    Grafica un bar chart comparando el R² global entre modelos.

    Parámetros
    ----------
    r2_dict : dict {nombre_modelo: valor_r2}
        Ej: {'Baseline CNN': 0.72, 'U-Net': 0.85, 'Bob': 0.10}
    """
    nombres = list(r2_dict.keys())
    valores = list(r2_dict.values())
    colores = ['steelblue', '#722F37', 'seagreen'][:len(nombres)]

    fig, ax = plt.subplots(figsize=(7, 5))
    barras = ax.bar(nombres, valores, color=colores, edgecolor='black', width=0.5)

    # Línea de referencia en R²=0 (peor que predecir la media)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)

    # Valor encima de cada barra
    for barra, valor in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width() / 2,
                valor + 0.01,
                f'{valor:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_ylabel('R² (píxeles activos)')
    ax.set_title('R² Global por modelo')
    ax.set_ylim(min(0, min(valores)) - 0.1, max(valores) + 0.1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    ruta_guardado = generar_ruta_imagen('../images', nombre_figura)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica guardada en: {ruta_guardado}")
    plt.show()

def graficar_picos_de_error(df_metricas, modelo, dataset, nombre_modelo='Modelo', top_n=3, metrica='mse', canal_entrada=0, vmax_percentil=99):
    """
    Encuentra los timesteps (índices) donde el modelo tuvo el mayor error 
    (los picos más altos) y grafica la reconstrucción para entender qué pasó.
    
    Parámetros
    ----------
    df_metricas   : DataFrame devuelto por evaluar_dataset().
    modelo        : modelo PyTorch ya cargado y en eval mode.
    dataset       : TormentaDataset (val_dataset).
    nombre_modelo : str, para el título.
    top_n         : int, cantidad de picos a mostrar (por defecto los 3 peores).
    metrica       : str, la columna del df a usar para buscar los picos ('mse', 'mae', etc).
    canal_entrada : int, índice del canal a mostrar en la gráfica de entrada.
    vmax_percentil : int, default 99
        Percentil de los valores activos (LIGHTDENS > 0) utilizado como
        límite superior de la escala de color en las reconstrucciones.
        Valores menores achican más la escala y resaltan mejor la estructura
        espacial a costa de saturar los picos más intensos.
    """
    # Verificamos que la métrica exista en el DataFrame
    if metrica not in df_metricas.columns:
        raise ValueError(f"La métrica '{metrica}' no se encuentra en el DataFrame.")

    # Calculamos la media para dar un poco de contexto de "qué tan grave" es el pico
    media_metrica = df_metricas[metrica].mean()
    
    # Extraemos las 'top_n' filas con los valores más altos de la métrica
    df_picos = df_metricas.nlargest(top_n, metrica)
    
    print(f"{"="*60}")
    print(f"🔍 ANALIZANDO LOS {top_n} PEORES CASOS SEGÚN {metrica.upper()}")
    print(f"Media global de {metrica.upper()} en este dataset: {media_metrica:.4f}")
    print(f"{"="*60}\n")
    
    # Iteramos sobre esas peores filas
    for ranking, (_, fila) in enumerate(df_picos.iterrows(), start=1):
        idx_pico = int(fila['idx'])
        valor_pico = fila[metrica]
        
        # Calculamos cuántas veces peor es este pico en comparación a la media
        multiplo = valor_pico / media_metrica if media_metrica > 0 else 0
        
        print(f"⚠️ PICO #{ranking} | Timestep: {idx_pico}")
        print(f"-> {metrica.upper()}: {valor_pico:.4f} (El error es {multiplo:.1f} veces la media global)")
        
        # Graficamos el índice extraído
        graficar_reconstruccion(
            modelo=modelo,
            dataset=dataset,
            nombre_modelo=nombre_modelo,
            canal_entrada=canal_entrada,
            idx=idx_pico,
            vmax_percentil=vmax_percentil
        )
        print("-" * 60)
        print("\n")

In [ ]:
archivos = sorted(glob.glob('../modelos/*baseline*.pth'))
checkpoint_baseline = torch.load(archivos[-1], map_location=device)
modelo_baseline.load_state_dict(checkpoint_baseline['best_model_state_dict'])
modelo_baseline.eval()
resumen_entrenamiento(checkpoint_baseline, 'Baseline CNN WMSE', train_loader, val_loader, BATCH_SIZE)

In [ ]:
graficar_curva_error(checkpoint_baseline, nombre_modelo='Baseline CNN WMSE')

In [ ]:
graficar_reconstruccion(modelo_baseline, val_dataset, nombre_modelo='Baseline CNN WMSE', idx=540)

In [ ]:
animar_reconstruccion(modelo_baseline, val_dataset, nombre_modelo='Baseline CNN WMSE')

In [ ]:
df_historial = mostrar_historial_tabular(checkpoint_baseline)
ruta_txt = "../modelos/historial_baseline_WMSE.txt"

with open(ruta_txt, 'w', encoding='utf-8') as f:
    f.write("=== HISTORIAL DE ENTRENAMIENTO ===\n")
    f.write(df_historial.to_string(index=False))
    
print(f"¡Guardado en {ruta_txt}!")

In [ ]:
archivos = sorted(glob.glob('../modelos/*unet_FACL*.pth'))
checkpoint_unet = torch.load(archivos[-1], map_location=device)
modelo_unet.load_state_dict(checkpoint_unet['best_model_state_dict'])
modelo_unet.eval()
resumen_entrenamiento(checkpoint_unet, 'U-Net', train_loader, val_loader, BATCH_SIZE)

In [ ]:
graficar_curva_error(checkpoint_unet, nombre_modelo='U-Net FACL')

In [ ]:
graficar_reconstruccion(modelo_unet, val_dataset, nombre_modelo='U-Net FACL')

In [ ]:
animar_reconstruccion(modelo_unet, val_dataset, nombre_modelo='U-Net FACL')

In [ ]:
df_historial = mostrar_historial_tabular(checkpoint_unet)
ruta_txt = "../modelos/historial_unet_FACL.txt"

with open(ruta_txt, 'w', encoding='utf-8') as f:
    f.write("=== HISTORIAL DE ENTRENAMIENTO ===\n")
    f.write(df_historial.to_string(index=False))
    
print(f"¡Guardado en {ruta_txt}!")

In [ ]:
archivos = sorted(glob.glob('../modelos/*bob*.pth'))
checkpoint_bob = torch.load(archivos[-1], map_location=device)
modelo_bob.load_state_dict(checkpoint_bob['best_model_state_dict'])
modelo_bob.eval()
resumen_entrenamiento(checkpoint_bob, 'Bob', train_loader, val_loader, BATCH_SIZE)

In [ ]:
graficar_curva_error(checkpoint_bob, nombre_modelo='Bob')

In [ ]:
df_historial = mostrar_historial_tabular(checkpoint_bob)
ruta_txt = "../modelos/historial_Bob.txt"

with open(ruta_txt, 'w', encoding='utf-8') as f:
    f.write("=== HISTORIAL DE ENTRENAMIENTO ===\n")
    f.write(df_historial.to_string(index=False))
    
print(f"¡Guardado en {ruta_txt}!")

In [ ]:
UMBRAL = 5.0  # Percentil 25 de LIGHTDENS > 0 en val_dataset

data_range_val = calcular_data_range_global(val_dataset)

#df_metricas_baseline = evaluar_dataset(modelo_baseline, val_dataset, umbral=UMBRAL, data_range=data_range_val)
df_metricas_unet     = evaluar_dataset(modelo_unet,     val_dataset, umbral=UMBRAL, data_range=data_range_val)
df_metricas_bob      = evaluar_dataset(modelo_bob,      val_dataset, umbral=UMBRAL, data_range=data_range_val)

#r2_baseline = calcular_r2_global(modelo_baseline, val_dataset)
r2_unet     = calcular_r2_global(modelo_unet,    val_dataset)
r2_bob      = calcular_r2_global(modelo_bob, val_dataset)
graficar_r2_comparativo({
#   'Baseline CNN WSE': r2_baseline,
    'U-Net WSE':        r2_unet,
    'Bob':          r2_bob,
})

In [ ]:
graficar_mse(df_metricas_baseline, nombre_modelo='Baseline CNN WMSE')

In [ ]:
graficar_rmse(df_metricas_baseline, nombre_modelo='Baseline CNN WMSE')

In [ ]:
graficar_mae(df_metricas_baseline, nombre_modelo='Baseline CNN WMSE')

In [ ]:
graficar_ssim(df_metricas_baseline, nombre_modelo='Baseline CNN WMSE')

In [ ]:
# graficar_hp(df_metricas_baseline, nombre_modelo='Baseline CNN WMSE')

In [ ]:
# graficar_csi(df_metricas_baseline, nombre_modelo='Baseline CNN WMSE')

In [ ]:
graficar_picos_de_error(
    df_metricas=df_metricas_baseline, 
    modelo=modelo_baseline, 
    dataset=val_dataset, 
    nombre_modelo='Baseline CNN WMSE', 
    top_n=7,           
    metrica='mse'
)

In [ ]:
graficar_mse(df_metricas_unet, nombre_modelo='U-Net FACL')

In [ ]:
graficar_rmse(df_metricas_unet, nombre_modelo='U-Net FACL')

In [ ]:
graficar_ssim(df_metricas_unet, nombre_modelo='U-Net FACL')

In [ ]:
# graficar_hp(df_metricas_unet, nombre_modelo='U-Net WMSE')

In [ ]:
# graficar_csi(df_metricas_unet, nombre_modelo='U-Net WMSE')

In [ ]:
graficar_picos_de_error(
    df_metricas=df_metricas_unet, 
    modelo=modelo_unet, 
    dataset=val_dataset, 
    nombre_modelo='U-Net FACL', 
    top_n=7,           
    metrica='mse'
)

In [ ]:
graficar_mse(df_metricas_bob, nombre_modelo='Bob')

In [ ]:
graficar_rmse(df_metricas_bob, nombre_modelo='Bob')

In [ ]:
graficar_ssim(df_metricas_bob, nombre_modelo='Bob')

In [ ]:
# graficar_hp(df_metricas_bob, nombre_modelo='Bob')

In [ ]:
# graficar_csi(df_metricas_bob , nombre_modelo='Bob')

## Determinar Umbral CSI y HP

In [ ]:
def barrido_umbrales(modelo, dataset, percentiles=[10, 25, 50, 75, 90], data_range=None):
    """
    Calcula CSI y HP para distintos umbrales, definidos como percentiles
    de la distribución de LIGHTDENS REAL condicionada a LIGHTDENS > 0.
    """
    # 1. Recolectar todos los valores reales > 0 de todo el dataset
    valores_positivos = []
    for idx in tqdm(range(len(dataset)), desc="Recolectando valores > 0"):
        _, y = dataset[idx]
        y_real = torch.clamp(torch.exp(y.squeeze()) - 0.1, min=0).numpy()
        valores_positivos.append(y_real[y_real > 0].flatten())
    valores_positivos = np.concatenate(valores_positivos)

    # 2. Calcular los umbrales correspondientes a cada percentil
    umbrales = {p: np.percentile(valores_positivos, p) for p in percentiles}
    print("Umbrales calculados (sobre LIGHTDENS > 0):")
    for p, u in umbrales.items():
        print(f"  Percentil {p}: {u:.4f} flash/columna")

    # 3. Para cada umbral, evaluar el dataset completo
    resultados = []
    for p, umbral in umbrales.items():
        df = evaluar_dataset(modelo, dataset, umbral=umbral, data_range=data_range)
        resultados.append({
            'percentil': p,
            'umbral': umbral,
            'csi_medio': df['csi'].mean(),
            'hp_medio': df['hp'].mean(),
        })

    return pd.DataFrame(resultados)


def graficar_barrido_umbrales(df_barrido, nombre_modelo='Modelo'):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(df_barrido['percentil'], df_barrido['csi_medio'], 'o-', label='CSI medio', color='#722F37')
    ax.plot(df_barrido['percentil'], df_barrido['hp_medio'],  'o-', label='HP medio', color='steelblue')
    ax.set_xlabel('Percentil de LIGHTDENS > 0 usado como umbral')
    ax.set_ylabel('Valor de la métrica')
    ax.set_title(f'Sensibilidad de CSI/HP al umbral — {nombre_modelo}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_barrido = barrido_umbrales(modelo_baseline, val_dataset, 
                               percentiles=[10, 25, 50, 75, 90], 
                               data_range=data_range_val)
graficar_barrido_umbrales(df_barrido, nombre_modelo='Baseline CNN WMSE')

In [ ]:
def graficar_prob_t(prob_slope=4e-4, max_prob=0.95, batches_por_epoca=None, epocas=60):
    steps = np.arange(0, epocas * batches_por_epoca)
    prob_t = np.minimum(steps * prob_slope * max_prob, max_prob)
    epocas_eje = steps / batches_por_epoca
    plt.plot(epocas_eje, prob_t)
    plt.xlabel("Época"); plt.ylabel("P(t) — prob. de usar FAL")
    plt.axhline(max_prob, ls='--', color='gray')
    plt.grid(alpha=0.3)
    plt.show()

graficar_prob_t(batches_por_epoca=len(train_loader), epocas=checkpoint_unet['epochs_entrenadas'])

In [ ]:
epocas=checkpoint_unet['epochs_entrenadas']
batches_por_epoca=len(train_loader)
steps = np.arange(0, epocas * batches_por_epoca)
prob_slope=4e-4
max_prob=0.95
prob_t = np.minimum(steps * prob_slope * max_prob, max_prob)
epoca_cruce = np.argmax(prob_t >= 0.5) / batches_por_epoca  # primera época donde P(t)>=0.5
plt.plot(checkpoint_unet['historial_val_loss'])
plt.axvline(epoca_cruce, color='red', ls='--', label='P(t)=0.5')
plt.legend()
plt.show()

In [ ]:
print(checkpoint_unet['mejor_epoca'])

In [ ]:
modelo_best = UNet_WRF().to(device)
modelo_best.load_state_dict(checkpoint_unet['best_model_state_dict'])

modelo_final = UNet_WRF().to(device)
modelo_final.load_state_dict(checkpoint_unet['model_state_dict'])

idx_test = 250  # el mismo que usaste en la imagen que me mostraste antes
graficar_reconstruccion(modelo_best,  val_dataset, nombre_modelo='U-Net (mejor época, ~cruce)', idx=idx_test)
graficar_reconstruccion(modelo_final, val_dataset, nombre_modelo='U-Net (época final, FAL dominante)', idx=idx_test)

In [ ]:
for idx_test in range(279):
    X, y = val_dataset[idx_test]
    with torch.no_grad():
        pred_best  = modelo_best(X.unsqueeze(0).to(device)).cpu().squeeze()
        pred_final = modelo_final(X.unsqueeze(0).to(device)).cpu().squeeze()
    y_real = val_dataset.desnormalizar_y(y.squeeze()).numpy()
    y_best = val_dataset.desnormalizar_y(pred_best).numpy()
    y_fin  = val_dataset.desnormalizar_y(pred_final).numpy()
    print(f"idx={idx_test} | real_max={y_real.max():.1f} | best_max={y_best.max():.1f} | final_max={y_fin.max():.1f}")

In [ ]:
reales, predichos_best, predichos_final = [], [], []
for idx in range(len(val_dataset)):
    X, y = val_dataset[idx]
    with torch.no_grad():
        pb = modelo_best(X.unsqueeze(0).to(device)).cpu().squeeze()
        pf = modelo_final(X.unsqueeze(0).to(device)).cpu().squeeze()
    y_real = val_dataset.desnormalizar_y(y.squeeze()).numpy()
    reales.append(y_real.max())
    predichos_best.append(val_dataset.desnormalizar_y(pb).numpy().max())
    predichos_final.append(val_dataset.desnormalizar_y(pf).numpy().max())

plt.scatter(reales, predichos_best, alpha=0.4, label='Mejor época (23)', s=15)
plt.scatter(reales, predichos_final, alpha=0.4, label='Época final', s=15)
plt.plot([0, max(reales)], [0, max(reales)], 'k--', label='Predicción perfecta')
plt.xlabel('LIGHTDENS real (máx por timestep)')
plt.ylabel('LIGHTDENS predicho (máx por timestep)')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
for idx_test in [50, 140, 160, 200, 250]:
    X, y = val_dataset[idx_test]
    with torch.no_grad():
        pred_norm = modelo_final(X.unsqueeze(0).to(device)).cpu().squeeze()
    print(f"idx={idx_test} | pred_norm max={pred_norm.max().item():.4f} | pred_norm mean={pred_norm.mean().item():.4f}")

In [ ]:
def error_dc(y_real, y_pred):
    """Error en el componente DC del espectro = error en la suma total de la imagen."""
    dc_real = y_real.sum()
    dc_pred = y_pred.sum()
    return (dc_pred - dc_real) ** 2, dc_real, dc_pred

for idx_test in [50, 140, 160, 200, 250]:
    X, y_real = val_dataset[idx_test]
    with torch.no_grad():
        y_pred = modelo_final(X.unsqueeze(0).to(device)).cpu().squeeze()
    err, dc_r, dc_p = error_dc(y_real, y_pred)
    print(f"idx={idx_test} | DC real={dc_r:.1f} | DC pred={dc_p:.1f} | error²={err:.2e}")